# VinDr-Mammo Re-Stratification from Downloaded Files

## Purpose
Re-stratify already downloaded files to ensure proper 25% malignant / 75% benign distribution.

**Workflow:**
1. Load `download_progress.json` to get list of downloaded files
2. Match with metadata to get labels
3. Perform patient-level stratified sampling
4. Create new selection with proper distribution
5. Optionally move unused files to backup folder

---

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
from collections import defaultdict

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

## Step 2: Load Downloaded Files from Progress JSON

In [ ]:
# Configuration
BASE_DIR = Path('/content/drive/MyDrive/vindr-mammo')
PROGRESS_FILE = BASE_DIR / 'download_progress.json'
METADATA_FILE = BASE_DIR / 'metadata' / 'breast-level_annotations.csv'

# Target distribution
TARGET_TOTAL = 1000  # Use all available files (will auto-adjust to actual count)
TARGET_MALIGNANT_PCT = 0.25
TARGET_BENIGN_PCT = 0.75
RANDOM_SEED = 42

print("🎯 Re-Stratification Configuration")
print("=" * 70)
print(f"  Base directory: {BASE_DIR}")
print(f"  Target total files: {TARGET_TOTAL}")
print(f"  Target malignant: {int(TARGET_TOTAL * TARGET_MALIGNANT_PCT)} ({TARGET_MALIGNANT_PCT*100:.0f}%)")
print(f"  Target benign: {int(TARGET_TOTAL * TARGET_BENIGN_PCT)} ({TARGET_BENIGN_PCT*100:.0f}%)")
print(f"  Random seed: {RANDOM_SEED}")

In [ ]:
# Load progress file
print("\n📂 Loading downloaded files from progress.json")
print("=" * 70)

with open(PROGRESS_FILE, 'r') as f:
    progress = json.load(f)

downloaded_files = progress.get('downloaded_files', [])
print(f"✅ Found {len(downloaded_files)} files in download_progress.json")

# Verify files actually exist
verified_files = []
for file_path in downloaded_files:
    full_path = BASE_DIR / file_path
    if full_path.exists() and full_path.stat().st_size > 0:
        verified_files.append(file_path)

print(f"✅ Verified {len(verified_files)} files exist on disk")

# Auto-adjust to use ALL available files
print(f"\n🎯 Target: {TARGET_TOTAL} files")
print(f"📁 Available: {len(verified_files)} files")

if len(verified_files) < TARGET_TOTAL:
    print(f"⚠️  Using ALL {len(verified_files)} available files (less than target)")
    print(f"   Will stratify these {len(verified_files)} files to maintain 25%/75% distribution")
else:
    print(f"✅ Sufficient files available, will select {TARGET_TOTAL} with stratification")

## Step 3: Match with Metadata and Create Labels

In [ ]:
# Load metadata
print("\n📊 Loading and processing metadata")
print("=" * 70)

metadata_df = pd.read_csv(METADATA_FILE)
print(f"✅ Loaded metadata: {len(metadata_df)} total records")

# Extract image IDs from file paths
# Format: images/study_id/image_id.dicom
downloaded_records = []

for file_path in verified_files:
    parts = file_path.split('/')
    if len(parts) == 3 and parts[0] == 'images':
        study_id = parts[1]
        image_id = parts[2].replace('.dicom', '')
        downloaded_records.append({
            'file_path': file_path,
            'study_id': study_id,
            'image_id': image_id
        })

downloaded_df = pd.DataFrame(downloaded_records)
print(f"✅ Parsed {len(downloaded_df)} file paths")

# Merge with metadata
merged_df = downloaded_df.merge(
    metadata_df,
    on=['study_id', 'image_id'],
    how='left'
)

print(f"✅ Matched {len(merged_df)} files with metadata")

# Check for unmatched
unmatched = merged_df[merged_df['breast_birads'].isna()]
if len(unmatched) > 0:
    print(f"⚠️  WARNING: {len(unmatched)} files could not be matched with metadata")

# Filter out unmatched
merged_df = merged_df[merged_df['breast_birads'].notna()].copy()
print(f"✅ Final matched dataset: {len(merged_df)} files")

In [ ]:
# Create labels from BI-RADS
print("\n🏥 Processing BI-RADS labels")
print("=" * 70)

# Extract numeric BI-RADS
merged_df['birads_numeric'] = merged_df['breast_birads'].str.extract(r'(\d+)')[0].astype(float)

# Exclude BI-RADS 3 (probably benign - uncertain)
before_filter = len(merged_df)
merged_df = merged_df[merged_df['birads_numeric'] != 3].copy()
print(f"✅ Excluded BI-RADS 3: {before_filter} → {len(merged_df)} files")

# Create binary labels
# 0 = Benign (BI-RADS 1, 2)
# 1 = Malignant (BI-RADS 4, 5, 6)
merged_df['label'] = merged_df['birads_numeric'].apply(
    lambda x: 1 if x in [4, 5, 6] else 0
)

# Current distribution
label_counts = merged_df['label'].value_counts()
total = len(merged_df)
malignant = label_counts.get(1, 0)
benign = label_counts.get(0, 0)

print(f"\n📊 Current Distribution (Before Re-Stratification):")
print(f"  Malignant: {malignant:4d} ({malignant/total*100:5.1f}%)")
print(f"  Benign:    {benign:4d} ({benign/total*100:5.1f}%)")
print(f"  Total:     {total:4d}")

malignant_df = merged_df[merged_df['label'] == 1]
benign_df = merged_df[merged_df['label'] == 0]

print(f"\n  Available for sampling:")
print(f"    Malignant patients: {malignant_df['study_id'].nunique()}")
print(f"    Benign patients:    {benign_df['study_id'].nunique()}")

# Adjust target to use ALL available files (after filtering BI-RADS 3)
FINAL_TARGET = min(TARGET_TOTAL, total)
if FINAL_TARGET < TARGET_TOTAL:
    print(f"\n⚠️  Only {FINAL_TARGET} files available after filtering")
    print(f"   Will use ALL {FINAL_TARGET} files with 25%/75% stratification")
else:
    print(f"\n✅ Will select {FINAL_TARGET} files with 25%/75% stratification")

## Step 4: Patient-Level Stratified Sampling

In [ ]:
# Patient-level stratified sampling
print("\n🎯 Performing Patient-Level Stratified Sampling")
print("=" * 70)

np.random.seed(RANDOM_SEED)

# Target counts (use FINAL_TARGET from previous cell)
target_malignant = int(FINAL_TARGET * TARGET_MALIGNANT_PCT)
target_benign = int(FINAL_TARGET * TARGET_BENIGN_PCT)

print(f"\n  Targets:")
print(f"    Malignant: {target_malignant} files")
print(f"    Benign:    {target_benign} files")

# Get patient sizes
malignant_patient_sizes = malignant_df.groupby('study_id').size().reset_index(name='image_count')
benign_patient_sizes = benign_df.groupby('study_id').size().reset_index(name='image_count')

# Shuffle patients
malignant_patient_sizes = malignant_patient_sizes.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
benign_patient_sizes = benign_patient_sizes.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Select malignant patients
selected_malignant_patients = []
malignant_count = 0

for _, row in malignant_patient_sizes.iterrows():
    if malignant_count >= target_malignant:
        break
    selected_malignant_patients.append(row['study_id'])
    malignant_count += row['image_count']

# Select benign patients
selected_benign_patients = []
benign_count = 0

for _, row in benign_patient_sizes.iterrows():
    if benign_count >= target_benign:
        break
    selected_benign_patients.append(row['study_id'])
    benign_count += row['image_count']

print(f"\n  Selected:")
print(f"    Malignant: {len(selected_malignant_patients)} patients → {malignant_count} files")
print(f"    Benign:    {len(selected_benign_patients)} patients → {benign_count} files")

# Filter to selected patients
malignant_selected = malignant_df[malignant_df['study_id'].isin(selected_malignant_patients)]
benign_selected = benign_df[benign_df['study_id'].isin(selected_benign_patients)]

# Combine
stratified_df = pd.concat([malignant_selected, benign_selected], ignore_index=True)

print(f"\n✅ Stratified Selection Complete:")
print(f"  Total files: {len(stratified_df)}")
print(f"  Malignant:   {len(malignant_selected)} ({len(malignant_selected)/len(stratified_df)*100:.1f}%)")
print(f"  Benign:      {len(benign_selected)} ({len(benign_selected)/len(stratified_df)*100:.1f}%)")
print(f"  Patients:    {stratified_df['study_id'].nunique()}")

## Step 5: Save Stratified Selection

In [ ]:
# Save stratified selection
print("\n💾 Saving Stratified Selection")
print("=" * 70)

STRATIFIED_SELECTION_FILE = BASE_DIR / 'metadata' / 'stratified_selection.csv'
stratified_df.to_csv(STRATIFIED_SELECTION_FILE, index=False)
print(f"✅ Saved to: {STRATIFIED_SELECTION_FILE}")

# Also save just the file paths for easy reference
STRATIFIED_FILES_LIST = BASE_DIR / 'metadata' / 'stratified_files.txt'
with open(STRATIFIED_FILES_LIST, 'w') as f:
    for file_path in stratified_df['file_path']:
        f.write(f"{file_path}\n")
print(f"✅ Saved file list to: {STRATIFIED_FILES_LIST}")

# Create summary JSON
SUMMARY_FILE = BASE_DIR / 'metadata' / 'stratification_summary.json'
summary = {
    'total_files': len(stratified_df),
    'malignant_files': len(malignant_selected),
    'benign_files': len(benign_selected),
    'malignant_percentage': round(len(malignant_selected)/len(stratified_df)*100, 2),
    'benign_percentage': round(len(benign_selected)/len(stratified_df)*100, 2),
    'unique_patients': int(stratified_df['study_id'].nunique()),
    'malignant_patients': int(malignant_selected['study_id'].nunique()),
    'benign_patients': int(benign_selected['study_id'].nunique()),
    'random_seed': RANDOM_SEED,
    'target_distribution': {'malignant': TARGET_MALIGNANT_PCT, 'benign': TARGET_BENIGN_PCT}
}

with open(SUMMARY_FILE, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Saved summary to: {SUMMARY_FILE}")

## Step 6: Detailed Analysis of Stratified Selection

In [ ]:
# Detailed analysis
print("\n" + "=" * 80)
print("📊 STRATIFIED SELECTION ANALYSIS")
print("=" * 80)

print("\n🎯 Label Distribution:")
print(f"  Malignant: {len(malignant_selected):4d} ({len(malignant_selected)/len(stratified_df)*100:5.1f}%)")
print(f"  Benign:    {len(benign_selected):4d} ({len(benign_selected)/len(stratified_df)*100:5.1f}%)")
print(f"  Total:     {len(stratified_df):4d}")
print(f"\n  Target: {TARGET_MALIGNANT_PCT*100:.0f}% malignant / {TARGET_BENIGN_PCT*100:.0f}% benign")

deviation_mal = abs(len(malignant_selected)/len(stratified_df)*100 - TARGET_MALIGNANT_PCT*100)
deviation_ben = abs(len(benign_selected)/len(stratified_df)*100 - TARGET_BENIGN_PCT*100)
print(f"  Deviation: {deviation_mal:.1f}% malignant, {deviation_ben:.1f}% benign")

if deviation_mal < 3 and deviation_ben < 3:
    print("  ✅ EXCELLENT: Within 3% of target")
elif deviation_mal < 5 and deviation_ben < 5:
    print("  ✅ GOOD: Within 5% of target")
else:
    print("  ⚠️  ACCEPTABLE: Slight deviation from target")

print("\n👥 Patient Distribution:")
print(f"  Total patients:     {stratified_df['study_id'].nunique()}")
print(f"  Malignant patients: {malignant_selected['study_id'].nunique()}")
print(f"  Benign patients:    {benign_selected['study_id'].nunique()}")

images_per_patient = stratified_df.groupby('study_id').size()
print(f"\n  Images per patient:")
print(f"    Mean:   {images_per_patient.mean():.1f}")
print(f"    Median: {images_per_patient.median():.1f}")
print(f"    Min:    {images_per_patient.min()}")
print(f"    Max:    {images_per_patient.max()}")

print("\n🏥 BI-RADS Distribution:")
birads_counts = stratified_df['breast_birads'].value_counts().sort_index()
for birads, count in birads_counts.items():
    pct = count / len(stratified_df) * 100
    label = "Malignant" if birads in ['BI-RADS 4', 'BI-RADS 5', 'BI-RADS 6'] else "Benign"
    print(f"  {birads:15s}: {count:3d} ({pct:5.1f}%) [{label}]")

if 'laterality' in stratified_df.columns:
    print("\n🔄 Laterality Distribution:")
    laterality_counts = stratified_df['laterality'].value_counts()
    for side, count in laterality_counts.items():
        pct = count / len(stratified_df) * 100
        print(f"  {side:4s}: {count:3d} ({pct:5.1f}%)")

if 'view_position' in stratified_df.columns:
    print("\n📸 View Position Distribution:")
    view_counts = stratified_df['view_position'].value_counts()
    for view, count in view_counts.items():
        pct = count / len(stratified_df) * 100
        print(f"  {view:4s}: {count:3d} ({pct:5.1f}%)")

print("\n" + "=" * 80)
print("✅ READY FOR ML TRAINING")
print("=" * 80)
print(f"\nYour stratified dataset: {len(stratified_df)} files")
print(f"Location: {BASE_DIR / 'images'}")
print(f"Metadata: {STRATIFIED_SELECTION_FILE}")
print("\nNext steps:")
print("  1. Use stratified_selection.csv for your training pipeline")
print("  2. Implement patient-level train/val/test split")
print("  3. Start model development!")
print("\n" + "=" * 80)

## Optional: Move Non-Selected Files to Backup

**WARNING:** This will move files not in the stratified selection to a backup folder.
Only run this if you want to clean up your dataset!

In [ ]:
# OPTIONAL: Uncomment to backup non-selected files
# WARNING: This moves files!

# BACKUP_DIR = BASE_DIR / 'backup_non_selected'
# BACKUP_DIR.mkdir(exist_ok=True)

# selected_files = set(stratified_df['file_path'])
# all_files = set(verified_files)
# non_selected = all_files - selected_files

# print(f"Moving {len(non_selected)} non-selected files to backup...")

# for file_path in non_selected:
#     src = BASE_DIR / file_path
#     dst = BACKUP_DIR / file_path
#     dst.parent.mkdir(parents=True, exist_ok=True)
#     shutil.move(str(src), str(dst))

# print(f"✅ Moved {len(non_selected)} files to {BACKUP_DIR}")

In [ ]:
# Balanced visualization: 10 malignant + 10 benign
print("=" * 80)
print("📊 BALANCED VISUALIZATION: 10 MALIGNANT + 10 BENIGN")
print("=" * 80)

# Select 10 from each class
malignant_samples = stratified_df[stratified_df['label'] == 1].head(10)
benign_samples = stratified_df[stratified_df['label'] == 0].head(10)

# Combine: first 10 malignant, then 10 benign
balanced_samples = pd.concat([malignant_samples, benign_samples], ignore_index=True)

print(f"\nDisplaying {len(balanced_samples)} samples:")
print(f"  First 10:  MALIGNANT")
print(f"  Last 10:   BENIGN")

# Create grid: 4 rows x 5 columns
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for idx, (_, row) in enumerate(balanced_samples.iterrows()):
    if idx >= 20:
        break
    
    # Build file path
    file_path = BASE_DIR / row['file_path']
    
    # Load DICOM
    try:
        dcm = pydicom.dcmread(str(file_path))
        img = dcm.pixel_array
        
        # Handle MONOCHROME1 (inverted grayscale)
        if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == "MONOCHROME1":
            img = img.max() - img
        
        # Display image
        axes[idx].imshow(img, cmap='gray')
        
        # Create title with label info
        label_name = "MALIGNANT" if row['label'] == 1 else "BENIGN"
        birads = row['breast_birads']
        laterality = row.get('laterality', 'N/A')
        view = row.get('view_position', 'N/A')
        patient_id = row['study_id'][:8]  # First 8 chars of patient ID
        
        title = f"[{idx+1}] {label_name}\n{birads}\n{laterality} {view} | {patient_id}"
        
        # Color code: red for malignant, green for benign
        color = 'red' if row['label'] == 1 else 'green'
        axes[idx].set_title(title, fontsize=9, color=color, fontweight='bold')
        
        # Add border around image
        for spine in axes[idx].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
        
    except Exception as e:
        # If file can't be loaded, show error
        axes[idx].text(0.5, 0.5, f"Error loading\nimage {idx+1}\n{str(e)[:30]}", 
                      ha='center', va='center', fontsize=8, color='red')
        axes[idx].set_title(f"[{idx+1}] ERROR", color='red')
    
    axes[idx].axis('off')

# Hide any unused subplots
for idx in range(len(balanced_samples), 20):
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle('Balanced Sample: 10 Malignant (Red) + 10 Benign (Green)', 
             fontsize=16, fontweight='bold', y=1.00)
plt.show()

print("\n✅ Balanced visualization complete!")
print("\nLegend:")
print("  🔴 Red titles/borders  = MALIGNANT (BI-RADS 4, 5, 6)")
print("  🟢 Green titles/borders = BENIGN (BI-RADS 1, 2)")
print("=" * 80)

## 📊 Balanced Visualization: 10 Malignant + 10 Benign

Display equal numbers of malignant and benign samples for comparison

In [ ]:
# Install pydicom if needed
try:
    import pydicom
except ImportError:
    !pip install -q pydicom
    import pydicom

import matplotlib.pyplot as plt
import numpy as np

# Visualize top 20 samples
print("=" * 80)
print("📸 VISUALIZING TOP 20 SAMPLES")
print("=" * 80)

# Get top 20 samples (mix of malignant and benign)
top_20 = stratified_df.head(20).copy()

print(f"\nDisplaying {len(top_20)} samples:")
mal_count = (top_20['label'] == 1).sum()
ben_count = (top_20['label'] == 0).sum()
print(f"  Malignant: {mal_count}")
print(f"  Benign:    {ben_count}")

# Create grid: 4 rows x 5 columns
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for idx, (_, row) in enumerate(top_20.iterrows()):
    if idx >= 20:
        break
    
    # Build file path
    file_path = BASE_DIR / row['file_path']
    
    # Load DICOM
    try:
        dcm = pydicom.dcmread(str(file_path))
        img = dcm.pixel_array
        
        # Handle MONOCHROME1 (inverted grayscale)
        if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == "MONOCHROME1":
            img = img.max() - img
        
        # Display image
        axes[idx].imshow(img, cmap='gray')
        
        # Create title with label info
        label_name = "MALIGNANT" if row['label'] == 1 else "BENIGN"
        birads = row['breast_birads']
        laterality = row.get('laterality', 'N/A')
        view = row.get('view_position', 'N/A')
        
        title = f"[{idx+1}] {label_name}\n{birads} | {laterality} {view}"
        
        # Color code: red for malignant, green for benign
        color = 'red' if row['label'] == 1 else 'green'
        axes[idx].set_title(title, fontsize=10, color=color, fontweight='bold')
        
    except Exception as e:
        # If file can't be loaded, show error
        axes[idx].text(0.5, 0.5, f"Error loading\nimage {idx+1}\n{str(e)[:30]}", 
                      ha='center', va='center', fontsize=8, color='red')
        axes[idx].set_title(f"[{idx+1}] ERROR", color='red')
    
    axes[idx].axis('off')

# Hide any unused subplots
for idx in range(len(top_20), 20):
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle('Top 20 Samples from Stratified Selection', fontsize=16, fontweight='bold', y=1.00)
plt.show()

print("\n✅ Visualization complete!")
print("=" * 80)

## 📸 Visualize Top 20 Samples

Display the first 20 images from the stratified selection (mix of malignant and benign)